# Prompt Tuning — Llama 3.2 3B on Bitext Customer Support

**What gets trained:** 20 soft token embeddings (shape: 20 × 4096 = 81,920 params)  
**Base model:** Fully frozen — not a single weight changes  
**Method:** PEFT PromptTuningConfig + HuggingFace Trainer  

> Note: Unsloth is intentionally not used here. Unsloth's kernels and grad scaler  
> are built around LoRA and break when PEFT injects virtual tokens instead.

In [1]:
# Cell 1 — Install
!pip install transformers datasets peft bitsandbytes accelerate trl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.5 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


In [5]:
# Cell 3 — Config
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH  = 512
NUM_SOFT_TOKENS = 20

In [6]:
# Cell 4 — Load tokenizer and model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"Total params : {total_params:,}")
print(f"Hidden size  : {model.config.hidden_size}")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Model loaded
Total params : 1,803,463,680
Hidden size  : 3072


In [ ]:
# Cell 5 — Wrap model with PEFT Prompt Tuning
#
# PromptTuningInit.TEXT initialises the 20 soft tokens from a real sentence
# rather than random noise — faster convergence since we start in a
# meaningful region of the embedding space instead of random coordinates.

peft_config = PromptTuningConfig(
    task_type               = "CAUSAL_LM",
    num_virtual_tokens      = NUM_SOFT_TOKENS,
    prompt_tuning_init      = PromptTuningInit.TEXT,
    prompt_tuning_init_text = "You are a helpful customer support agent. Answer the customer's questions politely and professionally.",
    tokenizer_name_or_path  = MODEL_NAME,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 61,440 || all params: 3,212,811,264 || trainable%: 0.0019


In [8]:
# Cell 6 — Load and format Bitext dataset
raw_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)

def format_prompt(examples):
    texts = []
    for instruction, response in zip(examples["instruction"], examples["response"]):
        text = (
            f"<|begin_of_text|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{instruction}"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{response}"
            f"<|eot_id|>"
        )
        texts.append(text)
    return {"text": texts}

dataset = raw_dataset.map(format_prompt, batched=True)
print(f"Dataset : {len(dataset):,} examples")
print(f"Sample  : {dataset[0]['text'][:200]}...")

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset : 26,872 examples
Sample  : <|begin_of_text|><|start_header_id|>user<|end_header_id|>

question about cancelling order {{Order Number}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I've understood you have a question ...


In [9]:
# Cell 7 — Tokenize dataset
def tokenize(examples):
    tokens = tokenizer(
        examples["text"],
        truncation = True,
        max_length = MAX_SEQ_LENGTH,
        padding    = "max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    tokenize,
    batched     = True,
    remove_columns = dataset.column_names,
    num_proc    = 2,
)

print(f"Tokenized dataset : {len(tokenized_dataset):,} examples")
print(f"Columns           : {tokenized_dataset.column_names}")

Map (num_proc=2):   0%|          | 0/26872 [00:00<?, ? examples/s]

Tokenized dataset : 26,872 examples
Columns           : ['input_ids', 'attention_mask', 'labels']


In [10]:
# Cell 8 — Train
training_args = TrainingArguments(
    output_dir                  = "outputs",
    max_steps                   = 100,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 3e-2,
    warmup_steps                = 10,
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    logging_steps               = 1,
    seed                        = 42,
    report_to                   = "none",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss
1,2.134645
2,2.471758
3,2.306550
4,3.643174
5,6.327873
6,4.697290
7,4.229052
8,4.253168
9,2.913738
10,2.224664


TrainOutput(global_step=100, training_loss=1.5072292733192443, metrics={'train_runtime': 2974.2717, 'train_samples_per_second': 0.269, 'train_steps_per_second': 0.034, 'total_flos': 6927353590579200.0, 'train_loss': 1.5072292733192443, 'epoch': 0.02977076510866329})

In [11]:
# Cell 9 — Inference after training
model.eval()

test_questions = [
    "I need to cancel my order, how can I do that?",
    "I want to change the delivery address for my purchase.",
    "I haven't received my refund yet, can you help?",
    "How do I track my package?",
]

print("=" * 60)
print("AFTER TRAINING — Prompt Tuned Model Responses")
print("=" * 60)

for question in test_questions:
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}"
        f"<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 150,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    response   = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)

    print(f"\nQ: {question}")
    print(f"A: {response.strip()}")
    print("-" * 60)

AFTER TRAINING — Prompt Tuned Model Responses


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")



Q: I need to cancel my order, how can I do that?
A: Thank you for reaching out to us! I understand that you need assistance with canceling your order. To help you with this, please provide me with the order number and any other relevant details. Once I have this information, I can guide you through the process of canceling your order. Your satisfaction is our top priority, and I'm here to ensure that your experience with us is seamless and hassle-free. Please let me know if there's anything else I can assist you with.
------------------------------------------------------------

Q: I want to change the delivery address for my purchase.
A: I'm so glad you reached out! I'm here to help you with your request to change the delivery address for your purchase. To begin, could you please provide me with the new address you'd like to use? I'll make sure to update the information and ensure that your order is delivered to the correct location. Your convenience and satisfaction are our top prio

In [ ]:
# Cell 10 — Save
# Saves only the soft prompt weights (~160 KB)
# The base model is unchanged and never needs to be re-saved

model.save_pretrained("soft_prompt_bitext")
tokenizer.save_pretrained("soft_prompt_bitext")

import os
size_kb = sum(
    os.path.getsize(os.path.join("soft_prompt_bitext", f))
    for f in os.listdir("soft_prompt_bitext")
) / 1024
print(f"Saved to 'soft_prompt_bitext/' — {size_kb:.1f} KB total")

In [ ]:
# Cell 11 — Inspect what the soft tokens learned
# Find the nearest real vocabulary token to each soft token vector
# These are hints — soft tokens don't map cleanly to words

soft_embeddings = model.prompt_encoder.default.embedding.weight.detach().float()
base_embeddings = model.base_model.get_input_embeddings().weight.detach().float()

print(f"Soft token matrix : {soft_embeddings.shape}")
print(f"Vocab matrix      : {base_embeddings.shape}")
print()

with torch.no_grad():
    for i, soft_vec in enumerate(soft_embeddings):
        sims     = torch.cosine_similarity(soft_vec.unsqueeze(0), base_embeddings)
        top3     = sims.topk(3)
        tokens   = [tokenizer.decode([idx.item()]) for idx in top3.indices]
        scores   = top3.values.tolist()
        print(
            f"  s{i+1:02d}: "
            f"{tokens[0]!r:15} ({scores[0]:.3f})  "
            f"{tokens[1]!r:15} ({scores[1]:.3f})  "
            f"{tokens[2]!r:15} ({scores[2]:.3f})"
        )